In [ ]:
import numpy as np
import skimage
import utils
import pathlib

In [ ]:
def otsu_thresholding(im: np.ndarray) -> int:
    """
        Otsu's thresholding algorithm that segments an image into 1 or 0 (True or False)
        The function takes in a grayscale image and outputs a threshold value

        args:
            im: np.ndarray of shape (H, W) in the range [0, 255] (dtype=np.uint8)
        return:
            (int) the computed thresholding value
    """
    assert im.dtype == np.uint8
    # You can also define other helper functions
    histogram = np.bincount(im.ravel(), minlength=256).astype(float)
    probabilities = histogram / histogram.sum()
    levels = np.arange(256)
    cumulative_weight = np.cumsum(probabilities)
    cumulative_mean = np.cumsum(probabilities * levels)
    global_mean = cumulative_mean[-1]

    valid = (cumulative_weight > 0) & (cumulative_weight < 1)
    between_class_variance = np.zeros(256)
    between_class_variance[valid] = (
        (global_mean * cumulative_weight[valid] - cumulative_mean[valid]) ** 2
        / (cumulative_weight[valid] * (1 - cumulative_weight[valid]))
    )
    threshold = int(np.argmax(between_class_variance))

    return threshold


In [ ]:
if __name__ == "__main__":
    impaths_to_segment = [
        pathlib.Path("thumbprint.png"),
        pathlib.Path("rice-shaded.png")
    ]
    for impath in impaths_to_segment:
        im = utils.read_image(impath)
        threshold = otsu_thresholding(im)
        print("Found optimal threshold:", threshold)

        # Segment the image by threshold
        segmented_image = (im >= threshold)
        assert im.shape == segmented_image.shape, "Expected image shape ({}) to be same as thresholded image shape ({})".format(
                im.shape, segmented_image.shape)
        assert segmented_image.dtype == bool, "Expected thresholded image dtype to be bool. Was: {}".format(
                segmented_image.dtype)

        segmented_image = utils.to_uint8(segmented_image)

        save_path = "{}-segmented.png".format(impath.stem)
        utils.save_im(save_path, segmented_image)

        
